In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # put this at the very top of your script

from datasets import load_dataset, Dataset
from dataclasses import dataclass
from typing import Optional
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer
from torch.utils.data import DataLoader

from tools_llada import TopKSorter, TruthCollector, MaxCollector
from plugins_llada import \
    SaveKVPreviousPlugin_Disabled, SaveKVPreviousPlugin_Enabled,\
    CachePastKVPlugin_Disabled, CachePastKVPlugin_Enabled,\
    CacheAttnPlugin_Disabled, CacheAttnPlugin_Enabled,\
    CacheVOPlugin_Disabled, CacheVOPlugin_Enabled

from configs_llada import DiffusionConfig

from components_llada import SimpleLogitsSnapshot
from tools_llada import ConfKSorter, ConfCollectorInterface, BlockDiffusionQuotaHelper
from plugins_llada import InspectorPlugin

from dataset_llada import LIST_DATASET
from datapreprocess_llada import parse_lines_with_index, merge_subdocs, PATTEN_REG_WIKI
from dataprocess_llada import Tokenizer_wiki_simple, Collater_wiki_simple

from modeling_llada_yukai_v4 import LLaDAModelLM

from tools_debug import jprint


config = DiffusionConfig(
    id_model='GSAI-ML/LLaDA-8B-Base',
    len_prompt=128,
    len_target=256,
    num_blocks=4,
    num_unmask_per_step=1,
    id_mask=126336,
    size_batch=1,
    device='cuda:0',
    klass_sorter=TopKSorter,
    klass_collector=TruthCollector,
    klass_save_kv_previous=SaveKVPreviousPlugin_Disabled,
    klass_cache_past_kv=CachePastKVPlugin_Enabled,
    klass_cache_attn=CacheAttnPlugin_Enabled,
    klass_cache_vo=CacheVOPlugin_Disabled
)

/home/exx/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

'''load dataset first'''
name_dataset = LIST_DATASET[1]
ds = load_dataset(*name_dataset, split='all')
docs, _ = parse_lines_with_index(PATTEN_REG_WIKI, ds['text'])
docs = docs['subdocs']

samples = []
for doc in docs:
    lines_1 = doc['texts']
    paragraph_1 = ' '.join(lines_1)
    lines_remain, titles = merge_subdocs(doc['subdocs'])
    paragraph_remain = ' '.join(lines_remain)
    prefix = paragraph_1
    target = paragraph_remain
    samples.append({'text': paragraph_1 + ' ' + paragraph_remain})
# end

ds_origin = Dataset.from_list(samples[:100])


'''initialize constant hyper-parameters'''

'''load tokenizer'''
tokenizer = AutoTokenizer.from_pretrained(
    config.id_model,
    trust_remote_code=True
)

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'
# end
assert tokenizer.pad_token_id != 126336


'''load model'''
model_kwargs = {}
model = LLaDAModelLM.from_pretrained(
    config.id_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    **model_kwargs
)

model = model.eval().to(config.device)


'''start to handle dataset based on hyper-parameter'''
len_max = config.len_prompt + config.len_target
ds = ds_origin\
    .filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)\
    .map(Tokenizer_wiki_simple(tokenizer, len_max), remove_columns=ds_origin.column_names)\
    .filter(lambda x: x["length"] >= len_max)\
    .sort("length")
# end

'''prepare dataloader'''
loader = DataLoader(
    ds,
    batch_size=config.size_batch,
    shuffle=False,                 # keep sorted order
    collate_fn=Collater_wiki_simple(len_max, config.len_prompt, config.len_target, config.id_mask),
    drop_last=False
)

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Filter: 100%|██████████| 100/100 [00:00<00:00, 14667.96 examples/s]


In [3]:
@ torch.no_grad()
def run_model_semi_cached_snapshot_refresh_one_rank_attn(model, x, y, config_diffusion, *args, **kwargs):

    '''declare required variables'''
    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.klass_sorter()
    collector = config_diffusion.klass_collector()

    plugin_cache_attn = kwargs['plugin_cache_attn']
    step_refresh_remainder = kwargs['step_refresh_remainder']
    budget_refresh = kwargs['budget_refresh']

    idx_refresh = torch.tensor([], dtype=torch.long, device=x.device)
    p_finalized = torch.zeros(x.shape, dtype=torch.float64, device=x.device)

    position_start, position_end = 0, len_prompt

    idx_denoising = torch.arange(position_start, position_end, dtype=torch.long, device=x.device)
    idx_current = torch.cat([idx_refresh, idx_denoising])
    shape_target = (x.shape[0], position_end, -1)
    logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
    snapshot = SimpleLogitsSnapshot(logits, x[:, idx_current], y[:, idx_current], id_mask)

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,position_start:position_end] == id_mask
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        idx_denoising = torch.arange(position_start, position_end, dtype=torch.long).to(x.device)
        idx_current = torch.cat([idx_refresh, idx_denoising]) 
        shape_target = (x.shape[0], position_end, -1)

        logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
        logits_denoising = logits[:, -idx_denoising.shape[-1]:]
        logits_accumulated = torch.cat([snapshot.get_logits(), logits_denoising], dim=1)
        x_accumulated = x[:, :position_end]
        y_accumulated = y[:, :position_end]
        snapshot = SimpleLogitsSnapshot(logits_accumulated, x_accumulated, y_accumulated, id_mask)

        for step in range(step_per_block):

            if step % step_refresh_remainder == 0 and step > 0:
                idx_denoising = torch.arange(position_start, position_end, dtype=torch.long).to(x.device)
                idx_current = torch.cat([idx_refresh, idx_denoising]) 
                shape_target = (x.shape[0], position_end, -1)                
                logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
                logits_denoising = logits[:, -idx_denoising.shape[-1]:]
                logits_accumulated = torch.cat([snapshot.get_logits()[:, :-logits_denoising.shape[1], :], logits_denoising], dim=1)
                x_accumulated = x[:, :position_end]
                y_accumulated = y[:, :position_end]
                snapshot = SimpleLogitsSnapshot(logits_accumulated, x_accumulated, y_accumulated, id_mask)

                conf_snapshot = snapshot.transform_logits(collector)    # 全的
                idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)    # 全的

                '''attn start to work'''

                num_unmask_total = sum([quota_helper.get_quota(step+i) for i in range(min(step_per_block - step, step_refresh_remainder))])
                idx_transform_total_2d = idx_sorted_by_conf[:, :num_unmask_total]     # 全的(2d)
                idx_transform_total_1d = idx_transform_total_2d.squeeze(0)

                scores_attn_all = plugin_cache_attn.collect_attn_from_all_blocks(model) # (B, Blk, K)
                idx_query = idx_transform_total_1d - position_start
                scores_attn = scores_attn_all[:, idx_query, :] #(B, Q, K))
                del scores_attn_all
                scores_key = scores_attn.mean(dim=(0, 1))          # (seq_k,) -> 3个? 有问题
                scores_key[idx_transform_total_1d] = 0.0
                scores_key = torch.where(torch.arange(idx_sorted_by_conf.shape[-1], device=x.device)>len_prompt, scores_key, 0.0)
                idx_most_attended_keys = torch.argsort(scores_key, descending=True)[:budget_refresh]

                idx_transform_previous_1d = idx_transform_2d.squeeze(0)
                # torch.Size([3]) torch.Size([1, 1]) torch.Size([1])
                # jprint(idx_most_attended_keys.shape, idx_transform_2d.shape, idx_transform_previous_1d.shape)
                idx_refresh = torch.cat([idx_most_attended_keys, idx_transform_previous_1d[-idx_transform_previous_1d.shape[0]:]])
                del idx_transform_previous_1d

            else:
                conf_snapshot = snapshot.transform_logits(collector)    # 全的
                idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)    # 全的   
            # end

            num_unmask = quota_helper.get_quota(step)
            idx_transform_2d = idx_sorted_by_conf[:, :num_unmask]     # 全的(2d)
            # jprint('idx_sorted_by_conf', idx_sorted_by_conf.shape)

            idx_current = torch.cat([idx_refresh, idx_transform_2d.squeeze(0)], dim=-1)
            logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
            logits_transform = logits[:, -idx_transform_2d.shape[-1]:]

            snapshot.update_logits_(idx_transform_2d, logits_transform)
            conf_snapshot = snapshot.transform_logits(collector)
            snapshot.materialize_by_idx_(idx_transform_2d, conf_snapshot)

            idx_refresh = idx_transform_2d.squeeze(0)
            idx_refresh = torch.cat([idx_refresh[:-idx_transform_2d.squeeze(0).shape[0]], idx_transform_2d.squeeze(0)])

            snapshot.update_this(1, idx_src=idx_transform_2d, y=x).update_this(1, idx_src=idx_transform_2d, p_finalized=p_finalized)
            
        # end for step
    # end for block

    return p_finalized[:, len_prompt:]
# end

In [4]:
import json
from tqdm import tqdm
from tools_llada import PPLCalculator, RefreshIdxHelper

filename = 'all_in_one_diff_128_256_4_abs_per_block_p_0326.json'
with open(filename, 'r') as file:
    data_refresh = json.load(file)
# end

refresher = RefreshIdxHelper(
    data_refresh,
    'v',
    config.size_block,
    randomed=False
)

calculator_ppl = PPLCalculator()
model\
    .fill_plugin(config.klass_cache_past_kv)\
    .fill_plugin(config.klass_save_kv_previous)\
    .fill_plugin(config.klass_cache_attn)\
    .fill_plugin(config.klass_cache_vo)

plugin_cache_past_kv = config.klass_cache_past_kv()
plugin_cache_attn = config.klass_cache_attn()

for budget_refresh in (2,4,8,16,32):
    for step_refresh_remainder in (2,3,4,6,8,10,12,15,20,25,30):
        '''start the evaluation process'''
        for id_batch, batch in enumerate(tqdm(loader)):
            plugin_cache_past_kv.clear(model)
            plugin_cache_attn.clear(model)

            conf = run_model_semi_cached_snapshot_refresh_one_rank_attn(
                model,
                batch['ids_prompt_masked_full'].to(config.device),
                batch['ids_target_masked_full'].to(config.device),
                config,
                plugin_cache_attn=plugin_cache_attn,
                step_refresh_remainder=step_refresh_remainder,
                budget_refresh=budget_refresh
            )

            print(f'step: {step_refresh_remainder}, budget: {budget_refresh} | {calculator_ppl.cal(conf)}')
            break
        # end
# end

  0%|          | 0/92 [00:14<?, ?it/s]


step: 2, budget: 2 | (8.744240975488603, 0.3675948835279633)


  0%|          | 0/92 [00:12<?, ?it/s]


step: 3, budget: 2 | (9.194766313852925, 0.3452280682364135)


  0%|          | 0/92 [00:11<?, ?it/s]


...

step: 20, budget: 32 | (11.981195844111909, 0.3362892699373713)


  0%|          | 0/92 [00:10<?, ?it/s]


step: 25, budget: 32 | (12.982522259554498, 0.30154275976516315)


  0%|          | 0/92 [00:10<?, ?it/s]

step: 30, budget: 32 | (12.727456396204321, 0.3190477790223847)
